# 04 — Test Random Forest, LightGBM, and XGBoost

Train the compact **Remove id + V** models from `02_ml_models.ipynb` and score `dataset/merged_test.parquet`.

If `merged_test` is still a copy of train, rebuild it with the cell below (IEEE test has no `isFraud` labels).

1. Fit RF / LightGBM / XGBoost on the first 80% of `merged_train` (temporal split).
2. Report labeled metrics on the last 20% of train.
3. Write probabilities for every row in `merged_test`.

Kernel: `ai` (LightGBM, XGBoost, scikit-learn).


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

try:
    import google.colab
    IS_COLAB = True
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    IS_COLAB = False

ROOT = Path("/content/drive/MyDrive/minor-thesis") if IS_COLAB else Path.cwd()
DATASET_PATH = ROOT / "dataset"
SAVED_PATH = ROOT / "saved"
print(f"Running on {'Google Colab' if IS_COLAB else 'Local'}")
print(f"Root: {ROOT}")


## Rebuild competition test if needed


In [ ]:
"""Rebuild dataset/merged_test.parquet from the IEEE test CSVs.

01_clean_dataset.ipynb accidentally wrote merged_train twice, so the current
merged_test.parquet is an identical copy of train (including isFraud).
The competition test file has no labels. Encoders are aligned to the existing
merged_train.parquet so RF / LightGBM / XGBoost / GNN features match.
"""

import gc
from pathlib import Path

import numpy as np
import pandas as pd

START_DATE = "2026-01-01"
MATCHING_BINARY = ["M1", "M2", "M3", "M5", "M6", "M7", "M8", "M9"]


def add_time_and_uid(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    dt = pd.to_datetime(START_DATE) + pd.to_timedelta(df["TransactionDT"], unit="s")
    df["DT_month"] = dt.dt.month
    df["DT_week"] = dt.dt.isocalendar().week.astype("int32")
    df["DT_day"] = dt.dt.day
    df["DT_weekday"] = dt.dt.weekday
    df["DT_hour"] = dt.dt.hour
    df["uid"] = (
        df["card1"].astype(str)
        + "_"
        + df["card2"].astype(str)
        + "_"
        + df["card3"].astype(str)
        + "_"
        + df["card5"].astype(str)
    )
    df["uid2"] = (
        df["uid"]
        + "_"
        + df["addr1"].astype(str)
        + "_"
        + df["P_emaildomain"].astype(str)
    )
    return df


def load_raw(split: str) -> pd.DataFrame:
    trans = pd.read_csv(DATASET_PATH / f"{split}_transaction.csv")
    ident = pd.read_csv(DATASET_PATH / f"{split}_identity.csv")
    ident.columns = [c.replace("-", "_") for c in ident.columns]
    df = trans.merge(ident, on="TransactionID", how="left")
    del trans, ident
    return add_time_and_uid(df)


def compact_feature_cols(columns) -> list[str]:
    cols = [
        c
        for c in columns
        if c not in {"isFraud", "TransactionID", "uid", "uid2"}
        and not str(c).startswith("id")
        and not str(c).startswith("V")
    ]
    for extra in ("uid", "uid2"):
        if extra in columns and extra not in cols:
            cols.append(extra)
    return cols


def test_is_train_copy(train: pd.DataFrame, test: pd.DataFrame) -> bool:
    if train.shape != test.shape:
        return False
    if "TransactionID" not in test.columns:
        return False
    return set(train["TransactionID"]) == set(test["TransactionID"])


def encode_like_train(raw_train: pd.DataFrame, encoded_train: pd.DataFrame, raw_test: pd.DataFrame) -> pd.DataFrame:
    ordered = [c for c in encoded_train.columns if c != "isFraud"]
    raw_train = raw_train.set_index("TransactionID")
    encoded_train = encoded_train.set_index("TransactionID")
    raw_test = raw_test.set_index("TransactionID")
    n = len(raw_test)
    data = {}

    for col in ordered:
        if col == "TransactionID":
            data[col] = raw_test.index.to_numpy()
            continue
        if col not in raw_test.columns:
            data[col] = np.full(n, -999, dtype=np.float32)
            continue

        if col in MATCHING_BINARY:
            data[col] = raw_test[col].map({"T": 1, "F": 0}).fillna(-1).to_numpy(dtype=np.int8)
            continue

        src = raw_train[col] if col in raw_train.columns else None
        is_cat = src is not None and (
            col in {"uid", "uid2"}
            or pd.api.types.is_object_dtype(src)
            or pd.api.types.is_string_dtype(src)
            or str(src.dtype) in {"category", "string"}
        )
        if is_cat:
            keys = src.fillna("Missing").astype(str)
            mapping = (
                pd.DataFrame({"k": keys.to_numpy(), "v": encoded_train[col].to_numpy()})
                .drop_duplicates("k")
                .set_index("k")["v"]
            )
            mapped = raw_test[col].fillna("Missing").astype(str).map(mapping)
            fill = int(encoded_train[col].max()) + 1 if encoded_train[col].notna().any() else -1
            data[col] = pd.to_numeric(mapped, errors="coerce").fillna(fill).to_numpy()
            continue

        data[col] = pd.to_numeric(raw_test[col], errors="coerce").fillna(-999).to_numpy()

    return pd.DataFrame(data)[ordered]


def reduce_like_train(df: pd.DataFrame, encoded_train: pd.DataFrame) -> pd.DataFrame:
    for col in df.columns:
        if col not in encoded_train.columns:
            continue
        dt = encoded_train[col].dtype
        arr = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=np.float64, copy=False)
        fill = -1.0 if pd.api.types.is_integer_dtype(dt) else -999.0
        arr = np.nan_to_num(arr, nan=fill, posinf=fill, neginf=fill)
        if pd.api.types.is_integer_dtype(dt):
            info = np.iinfo(dt)
            arr = np.clip(arr, info.min, info.max)
        df[col] = arr.astype(dt, copy=False)
    return df


def ensure_merged_test(force: bool = False) -> Path:
    train_path = DATASET_PATH / "merged_train.parquet"
    test_path = DATASET_PATH / "merged_test.parquet"
    encoded_train = pd.read_parquet(train_path)

    if test_path.exists() and not force:
        test = pd.read_parquet(test_path)
        if not test_is_train_copy(encoded_train, test) and "isFraud" not in test.columns:
            print(f"merged_test.parquet already looks like the competition test: {test.shape}")
            return test_path
        if test_is_train_copy(encoded_train, test):
            print(
                "WARNING: merged_test.parquet is an identical copy of merged_train "
                "(01_clean_dataset.ipynb saved train twice). "
                "Labeled test metrics on this file mix train and holdout rows. "
                "Re-run the test-parquet cell with force=True."
            )
            return test_path
        print(f"Using existing merged_test.parquet: {test.shape}")
        return test_path

    print("Loading raw train/test CSVs...")
    raw_train = load_raw("train")
    raw_test = load_raw("test")
    print(f"raw train {raw_train.shape}  raw test {raw_test.shape}")

    encoded = encode_like_train(raw_train, encoded_train, raw_test)
    del raw_train, raw_test
    gc.collect()
    encoded = reduce_like_train(encoded, encoded_train)
    encoded.to_parquet(test_path, index=False)
    print(f"Wrote {test_path}  shape={encoded.shape}  columns={len(encoded.columns)}")
    print("Test has no isFraud labels (IEEE-CIS competition test).")
    return test_path

ensure_merged_test(force=False)


## Fit and score

Hyperparameters match `02_ml_models.ipynb`:

- Random Forest: `n_estimators=300`, `class_weight='balanced'`
- LightGBM: `n_estimators=500`, `learning_rate=0.05`, `num_leaves=31`, `class_weight='balanced'`
- XGBoost: `n_estimators=500`, `learning_rate=0.05`, `max_depth=6`, `scale_pos_weight=n_neg/n_pos`


In [ ]:
"""Train RF / LightGBM / XGBoost on merged_train and score merged_test."""

import json
import warnings
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

DATASET_PATH = ROOT / "dataset"
SAVED_PATH = ROOT / "saved"
MODEL_DIR = SAVED_PATH / "ml_test_models"
PRED_DIR = SAVED_PATH / "test_predictions"
RANDOM_SEED = 42


def metrics_dict(name: str, y_true, y_pred, y_prob) -> dict:
    cm = confusion_matrix(y_true, y_pred)
    return {
        "Model": name,
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "Recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "ROC-AUC": float(roc_auc_score(y_true, y_prob)),
        "PR-AUC": float(average_precision_score(y_true, y_prob)),
        "Balanced Accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "MCC": float(matthews_corrcoef(y_true, y_pred)),
        "TN": int(cm[0, 0]),
        "FP": int(cm[0, 1]),
        "FN": int(cm[1, 0]),
        "TP": int(cm[1, 1]),
    }


def print_metrics(row: dict, y_true=None, y_pred=None) -> None:
    print(f"\n{'=' * 60}\n{row['Model']}\n{'=' * 60}")
    for key in [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "PR-AUC",
        "Balanced Accuracy",
        "MCC",
    ]:
        print(f"{key:20s}: {row[key]:.4f}")
    print(f"Confusion [{row['TN']}, {row['FP']}; {row['FN']}, {row['TP']}]")
    if y_true is not None:
        print(
            classification_report(
                y_true, y_pred, target_names=["Legitimate", "Fraud"], digits=4, zero_division=0
            )
        )


def make_models(y_train: pd.Series):
    pos = max(int((y_train == 1).sum()), 1)
    neg = int((y_train == 0).sum())
    return {
        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1,
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=-1,
            num_leaves=31,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=-1,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=6,
            random_state=RANDOM_SEED,
            n_jobs=-1,
            eval_metric="logloss",
            scale_pos_weight=neg / pos,
        ),
    }


def main():
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    PRED_DIR.mkdir(parents=True, exist_ok=True)
    ensure_merged_test(force=False)

    train = pd.read_parquet(DATASET_PATH / "merged_train.parquet")
    test = pd.read_parquet(DATASET_PATH / "merged_test.parquet")
    train = train.sort_values("TransactionDT").reset_index(drop=True)

    feature_cols = compact_feature_cols(train.columns)
    print(f"Train {train.shape}  Test {test.shape}")
    print(f"Features ({len(feature_cols)}): Remove id_* and V* (same compact set as v2 / GNN)")
    print(f"Test has isFraud: {'isFraud' in test.columns}")

    split = int(len(train) * 0.8)
    X = train[feature_cols]
    y = train["isFraud"]
    X_tr, X_va = X.iloc[:split], X.iloc[split:]
    y_tr, y_va = y.iloc[:split], y.iloc[split:]
    X_test = test[feature_cols]

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    holdout_rows = []
    pred_frame = pd.DataFrame({"TransactionID": test["TransactionID"]})

    for name, model in make_models(y_tr).items():
        print(f"\nFitting {name} on {len(X_tr):,} train rows...")
        model.fit(X_tr, y_tr)

        va_prob = model.predict_proba(X_va)[:, 1]
        va_pred = (va_prob >= 0.5).astype(np.int32)
        row = metrics_dict(f"{name} - train holdout 20%", y_va, va_pred, va_prob)
        print_metrics(row, y_va, va_pred)
        holdout_rows.append(row)

        te_prob = model.predict_proba(X_test)[:, 1]
        te_pred = (te_prob >= 0.5).astype(np.int32)
        pred_frame[f"{name}_prob"] = te_prob
        pred_frame[f"{name}_pred"] = te_pred
        print(f"Test predicted fraud rate ({name}): {te_pred.mean():.4f}  mean prob {te_prob.mean():.4f}")

        joblib.dump(model, MODEL_DIR / f"{name.lower()}_{timestamp}.pkl")

        if "isFraud" in test.columns:
            overlap = len(set(train["TransactionID"]) & set(test["TransactionID"]))
            if overlap:
                print(
                    f"WARNING: {overlap:,} TransactionIDs overlap train and test. "
                    "merged_test metrics are not a clean holdout."
                )
            te_row = metrics_dict(f"{name} - merged_test", test["isFraud"], te_pred, te_prob)
            print_metrics(te_row, test["isFraud"], te_pred)
            holdout_rows.append(te_row)

    metrics_df = pd.DataFrame(holdout_rows)
    metrics_df.to_parquet(PRED_DIR / f"ml_test_metrics_{timestamp}.parquet", index=False)
    metrics_df.to_csv(PRED_DIR / f"ml_test_metrics_{timestamp}.csv", index=False)
    pred_frame.to_parquet(PRED_DIR / f"ml_test_predictions_{timestamp}.parquet", index=False)
    pred_frame.to_csv(PRED_DIR / f"ml_test_predictions_{timestamp}.csv", index=False)

    feature_path = MODEL_DIR / f"features_{timestamp}.json"
    with open(feature_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "timestamp": timestamp,
                "feature_names": feature_cols,
                "n_train": int(len(X_tr)),
                "n_valid": int(len(X_va)),
                "n_test": int(len(X_test)),
                "test_has_labels": "isFraud" in test.columns,
            },
            f,
            indent=2,
        )

    print("\nHoldout metrics")
    print(metrics_df.to_string(index=False))
    print(f"\nSaved models -> {MODEL_DIR}")
    print(f"Saved predictions -> {PRED_DIR}")
    return metrics_df, pred_frame

metrics, preds = main()
display(metrics)
print(preds.head())


## Saved files

- `saved/ml_test_models/*.pkl` — fitted RF / LightGBM / XGBoost
- `saved/test_predictions/ml_test_metrics_*.csv` — labeled holdout metrics
- `saved/test_predictions/ml_test_predictions_*.csv` — test `TransactionID` plus probabilities
